# Phase 2: Data Preprocessing & Feature Engineering

This notebook handles the data preprocessing for the physics-constrained soil moisture ML pipeline. Specifically, it focuses on:
1. Loading the aligned daily time-series dataset from Phase 1.
2. Handling missing values and outliers.
3. Scaling/Normalizing numerical features.
4. Generating spatio-temporal sliding windows for sequential models (LSTM/Transformers).
5. Splitting the data into Train/Validation/Test sets.

In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Suppress unnecessary warnings
warnings.filterwarnings('ignore')

# Visualization styling
plt.style.use('default')
sns.set_theme(style="whitegrid")

print("Libraries successfully imported!")

Libraries successfully imported!


In [2]:
df = pd.read_csv('roi_aligned_timeseries_2021_2025.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"Initial dataset shape: {df.shape}")
print("Missing values per column:")
print(df.isnull().sum())

# Handle any edge-case missing values (e.g., fill forward then backward)
if df.isnull().values.any():
    df.fillna(method='ffill', inplace=True)
    df.fillna(method='bfill', inplace=True)
    print("Handled remaining missing values via forward/backward fill.")

# Set date as index for easier temporal slicing
df.set_index('date', inplace=True)
display(df.head())

Initial dataset shape: (1826, 17)
Missing values per column:
date                        0
potential_evaporation       0
sm_surface                187
temperature_2m              0
total_precipitation         0
B2                          0
B3                          0
B4                          0
B8                          0
NDVI                        0
VH                          0
VV                          0
clay                        0
constant                 1826
elevation                   0
sand                        0
slope                       0
dtype: int64
Handled remaining missing values via forward/backward fill.


,potential_evaporation,sm_surface,temperature_2m,total_precipitation,B2,B3,B4,B8,NDVI,VH,VV,clay,constant,elevation,sand,slope
date,,,,,,,,,,,,,,,,
2021-01-01,-0.064895,0.159273,287.775122,8.557963e-07,1144.752482,1332.601722,1166.394398,2386.942911,0.345933,-17.386736,-9.682761,27.116309,NaN,98.550338,37.425355,0.197415
2021-01-02,-0.059693,0.158111,289.226754,2.312732e-07,1144.752482,1332.601722,1166.394398,2386.942911,0.345933,-17.386736,-9.682761,27.116309,NaN,98.550338,37.425355,0.197415
2021-01-03,-0.059155,0.158204,290.845856,7.735890e-06,1144.752482,1332.601722,1166.394398,2386.942911,0.345933,-17.386736,-9.682761,27.116309,NaN,98.550338,37.425355,0.197415
2021-01-04,-0.060549,0.172161,292.214553,9.411331e-06,1106.706116,1268.856384,1182.136765,2244.874586,0.317548,-17.412232,-10.193458,27.116309,NaN,98.550338,37.425355,0.197415
2021-01-05,-0.042654,0.175980,292.534701,3.264632e-04,1068.659750,1205.111047,1197.879132,2102.806262,0.289162,-17.437728,-10.704155,27.116309,NaN,98.550338,37.425355,0.197415


In [3]:
target_col = 'sm_surface'
feature_cols = [c for c in df.columns if c != target_col]

# Initialize Scalers
X_scaler = MinMaxScaler()
y_scaler = MinMaxScaler() # Keeping separate scaler to easily inverse_transform predictions later

df_scaled = df.copy()
df_scaled[feature_cols] = X_scaler.fit_transform(df[feature_cols])
df_scaled[[target_col]] = y_scaler.fit_transform(df[[target_col]])

print("Features scaled successfully!")
display(df_scaled.head())

Features scaled successfully!


,potential_evaporation,sm_surface,temperature_2m,total_precipitation,B2,B3,B4,B8,NDVI,VH,VV,clay,constant,elevation,sand,slope
date,,,,,,,,,,,,,,,,
2021-01-01,0.834090,0.377790,0.103528,9.145119e-07,0.148402,0.125373,0.112387,0.092033,0.460140,0.495337,0.559440,0.0,NaN,0.0,0.0,0.0
2021-01-02,0.852868,0.374601,0.159958,2.471407e-07,0.148402,0.125373,0.112387,0.092033,0.460140,0.495337,0.559440,0.0,NaN,0.0,0.0,0.0
2021-01-03,0.854813,0.374858,0.222898,8.266643e-06,0.148402,0.125373,0.112387,0.092033,0.460140,0.495337,0.559440,0.0,NaN,0.0,0.0,0.0
2021-01-04,0.849778,0.413145,0.276104,1.005704e-05,0.141446,0.112547,0.115479,0.057479,0.413733,0.490359,0.463361,0.0,NaN,0.0,0.0,0.0
2021-01-05,0.914384,0.423621,0.288549,3.488616e-04,0.134490,0.099721,0.118571,0.022925,0.367326,0.485381,0.367282,0.0,NaN,0.0,0.0,0.0


In [4]:
def create_sequences(data, target_col_idx, seq_length):
    xs = []
    ys = []
    data_array = data.values
    
    for i in range(len(data_array) - seq_length):
        x = data_array[i:(i + seq_length)]
        y = data_array[i + seq_length, target_col_idx]
        xs.append(x)
        ys.append(y)
        
    return np.array(xs), np.array(ys)

SEQ_LENGTH = 14 # Look back 14 days
target_idx = df_scaled.columns.get_loc(target_col)

X, y = create_sequences(df_scaled, target_idx, SEQ_LENGTH)

print(f"X shape: {X.shape} (Samples, Sequence Length, Features)")
print(f"y shape: {y.shape} (Samples,)")

X shape: (1812, 14, 16) (Samples, Sequence Length, Features)
y shape: (1812,) (Samples,)


In [5]:
# Reconstruct the dates corresponding to the sequences
# Our sequence target 'y' is at index 'i + SEQ_LENGTH'
seq_dates = df_scaled.index[SEQ_LENGTH:]

# Create masks based on month
test_mask = seq_dates.month.isin([6, 7, 8, 9]) & (seq_dates.year == 2025)

train_val_mask = ~test_mask

# Separate Test Set (Monsoon)
X_test_season = X[test_mask]
y_test_season = y[test_mask]

# Separate Train/Val Sets (Dry/Winter seasons)
X_train_val = X[train_val_mask]
y_train_val = y[train_val_mask]

# Split the remaining dry/winter periods temporally into Train (80%) and Val (20%)
train_split_idx = int(len(X_train_val) * 0.8)
X_train, y_train = X_train_val[:train_split_idx], y_train_val[:train_split_idx]
X_val, y_val = X_train_val[train_split_idx:], y_train_val[train_split_idx:]

print(f"Training Set (Dry/Winter):   {X_train.shape[0]} samples")
print(f"Validation Set (Dry/Winter): {X_val.shape[0]} samples")
print(f"Test Set (Monsoon Holdout):  {X_test_season.shape[0]} samples")

# Convert to PyTorch tensors for Phase 3
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_season, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_season, dtype=torch.float32).unsqueeze(1)


Training Set (Dry/Winter):   1352 samples
Validation Set (Dry/Winter): 338 samples
Test Set (Monsoon Holdout):  122 samples
